# Librerias

In [60]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time

# Funciones

In [96]:
# 1. Función para buscar resultados en el JSON (recursiva)
def encontrar_resultados(obj):
    if isinstance(obj, dict):
        if "results" in obj and isinstance(obj["results"], list):
            if len(obj["results"]) > 0 and "productId" in obj["results"][0]:
                return obj["results"]
        for v in obj.values():
            res = encontrar_resultados(v)
            if res: return res
    elif isinstance(obj, list):
        for i in obj:
            res = encontrar_resultados(i)
            if res: return res
    return None

def scrap_tecnologia_falabella(max_pages=100):
    base_url = "https://www.falabella.com.pe/falabella-pe/category/CATG46221/Laptops-Gamer?f.product.L3_category_paths=cat40793%7C%7CTecnolog%C3%ADa%2FCATG34760%7C%7CZona+Gamer%2Fcat13000461%7C%7CComputaci%C3%B3n+gamer%2FCATG46221%7C%7CLaptops+Gamer"
    todos_los_productos = []
    
    print(f"Iniciando scraping de {max_pages} páginas...")

    for page in range(1, max_pages + 1):
        url = f"{base_url}&page={page}"
        print(f"Procesando página {page}...", end="\r")
        
        try:
            response = requests.get(
                url, 
                impersonate="chrome110",
                headers={
                    "Accept-Language": "es-PE,es;q=0.9",
                    "Referer": "https://www.google.com/"
                },
                timeout=30
            )

            if response.status_code != 200:
                print(f"\n[!] Error en página {page}: Status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            script_tag = soup.find("script", id="__NEXT_DATA__")

            if not script_tag:
                print(f"\n[!] No se halló __NEXT_DATA__ en página {page}")
                continue

            data = json.loads(script_tag.string)
            productos_pagina = encontrar_resultados(data)

            # --- CAMBIO CRÍTICO: VALIDACIÓN DE PARADA ---
            if not productos_pagina or len(productos_pagina) == 0:
                print(f"\n[!] Fin de catálogo alcanzado. No hay más productos en la página {page}.")
                break 
            # --------------------------------------------

            for p in productos_pagina:                
                precios_dict = {"CMR": None, "Internet": None, "Normal": None}
                for pr in p.get("prices", []):
                    tipo = pr.get("type", "").lower()
                    try:
                        valor = float(pr.get('price', [0])[0].replace(',', ''))
                    except:
                        valor = None
                    
                    if "cmr" in tipo: precios_dict["CMR"] = valor
                    elif "internet" in tipo or "event" in tipo: precios_dict["Internet"] = valor
                    elif "normal" in tipo: precios_dict["Normal"] = valor

                badge = p.get("discountBadge")

                # 2. Extraemos solo el número si el badge existe
                if badge and 'label' in badge:
                    try:
                        # Quitamos el '-', el '%' y convertimos a entero
                        porcentaje_limpio = int(badge['label'].replace('-', '').replace('%', '').strip())
                    except:
                        porcentaje_limpio = None
                else:
                    porcentaje_limpio = None

                todos_los_productos.append({
                    "Pagina": page,
                    "Marca": p.get("brand") if p.get("brand") else None,
                    "Nombre": p.get("displayName") if p.get("displayName") else None,
                    "Precio_CMR": precios_dict["CMR"],
                    "Precio_Internet": precios_dict["Internet"],
                    "Precio_Normal": precios_dict["Normal"],
                    "Porcentaje_Descuento_Internet_Oficial": porcentaje_limpio,
                    "Seller": p.get("sellerName") if p.get("sellerName") else None,
                    "Rating": p.get("rating") if p.get("rating") is not None else None,
                    "Reviews": p.get("totalReviews") if p.get("totalReviews") is not None else None,
                    "Es_Mejor_Vendedor": p.get("isBestSeller"),
                    "Es_Patrocinado": p.get("isSponsored"),
                    "ID_Producto": p.get("productId")
                })
            
            time.sleep(1.5) # Un poco más de tiempo para ser cautelosos

        except Exception as e:
            print(f"\n[!] Error crítico en página {page}: {e}")
            continue
            
    return pd.DataFrame(todos_los_productos)

# Exploración

In [97]:
# Ejecución
df = scrap_tecnologia_falabella(2)

# 3. Mostrar resumen y guardar
print(f"\n\nScraping finalizado exitosamente.")
print(f"Total de productos recolectados: {len(df)}")

# Guardar en CSV para no perder la data
#df.to_csv("productos_falabella_tecnologia.csv", index=False, encoding='utf-8-sig')

Iniciando scraping de 2 páginas...
Procesando página 2...

Scraping finalizado exitosamente.
Total de productos recolectados: 103


In [ ]:
df.head(5)

,Pagina,Marca,Nombre,Precio_CMR,Precio_Internet,Precio_Normal,Porcentaje_Descuento_Internet,Seller,Rating,Reviews,Es_Mejor_Vendedor,Es_Patrocinado,ID_Producto
0,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 506...,4899.0,4999.0,5699.0,14.0,FALABELLA,4.875,8,False,True,883746981
1,1,HP,"Laptop Gamer Omen 16-ap0003la, amd Ryzen Ai 7,...",NaN,7599.0,7999.0,5.0,FALABELLA,2,1,False,True,20936206
2,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 506...,4899.0,4999.0,5699.0,14.0,FALABELLA,4.875,8,False,False,883746981
3,1,LENOVO,Laptop Gamer Loq Intel Core I5 Rtx 4050 16gb R...,3199.0,3299.0,4299.0,26.0,FALABELLA,4.25,24,False,False,883668659
4,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 505...,3899.0,4099.0,4999.0,22.0,FALABELLA,4.6,10,False,False,883746941


In [ ]:
#Porcentaje de nulos en cada columna
print("\nPorcentaje de valores nulos por columna:")
print(df.isnull().mean() * 100)


Porcentaje de valores nulos por columna:
Pagina                            0.000000
Marca                             0.000000
Nombre                            0.000000
Precio_CMR                       83.653846
Precio_Internet                   0.000000
Precio_Normal                     0.961538
Porcentaje_Descuento_Internet     0.961538
Seller                            0.000000
Rating                           53.846154
Reviews                          53.846154
Es_Mejor_Vendedor                 0.000000
Es_Patrocinado                    0.000000
ID_Producto                       0.000000
dtype: float64


In [ ]:
df.describe()

,Pagina,Precio_CMR,Precio_Internet,Precio_Normal,Porcentaje_Descuento_Internet
count,104.000000,17.000000,104.000000,103.000000,103.000000
mean,1.461538,4540.176471,5272.844231,6719.319320,19.786408
std,0.500933,2286.047031,3641.606797,5309.637289,10.248530
min,1.000000,2899.000000,1350.000000,1999.000000,2.000000
25%,1.000000,3399.000000,3229.000000,3825.000000,11.500000
50%,1.000000,3949.000000,4074.000000,4999.000000,20.000000
75%,2.000000,4499.000000,6124.000000,7339.500000,26.000000
max,2.000000,12799.000000,25889.000000,43149.000000,53.000000


In [98]:
df["Porcentaje_Descuento_Internet_Calculado"] = ((df["Precio_Normal"] - df["Precio_Internet"]) / df["Precio_Normal"] * 100).round(2)
df["Fecha_Extraccion"] = pd.to_datetime("today").strftime("%Y-%m-%d")

In [99]:
df.head(5)

,Pagina,Marca,Nombre,Precio_CMR,Precio_Internet,Precio_Normal,Porcentaje_Descuento_Internet_Oficial,Seller,Rating,Reviews,Es_Mejor_Vendedor,Es_Patrocinado,ID_Producto,Porcentaje_Descuento_Internet_Calculado,Fecha_Extraccion
0,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 506...,4899.0,4999.0,5699.0,14.0,FALABELLA,4.875,8,False,True,883746981,12.28,2026-04-04
1,1,HP,"Laptop Gamer Omen 16-ap0003la, amd Ryzen Ai 7,...",NaN,7599.0,7999.0,5.0,FALABELLA,2,1,False,True,20936206,5.00,2026-04-04
2,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 506...,4899.0,4999.0,5699.0,14.0,FALABELLA,4.875,8,False,False,883746981,12.28,2026-04-04
3,1,LENOVO,Laptop Gamer Loq Intel Core I5 Rtx 4050 16gb R...,3199.0,3299.0,4299.0,26.0,FALABELLA,4.25,24,False,False,883668659,23.26,2026-04-04
4,1,LENOVO,Laptop Gamer LOQ serie 200 AMD Ryzen 7 RTX 505...,3899.0,4099.0,4999.0,22.0,FALABELLA,4.6,10,False,False,883746941,18.00,2026-04-04
